# 技能5 · Day 1 上机：用真实库构建营销 ReAct Agent

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库说明）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 LangChain/LangGraph 真实库构建带工具调用和记忆的 ReAct Agent
2. 实现 Reflection（评估者-优化者）循环，让 Agent 自检并改进输出
3. 对比 ReAct 与 Plan-Execute 两种架构模式在营销任务上的差异

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：LangChain + LangGraph（生产级 Agent 框架）。营销映射见下。

## 前置条件
- Python 3.10+
- 需要 LLM API Key（OpenAI 或 Anthropic），详见 data/README.md
- TODO1（定义工具）不需要 API Key，可直接测试

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

In [ ]:
# !pip install langgraph langchain langchain-openai -q
# 如使用 Anthropic Claude：# !pip install langchain-anthropic -q

## 1. 营销任务与 Agent 架构映射

**任务**：为一款护肤品（透肌焕亮精华液）制定营销策略，Agent 需要调用工具完成分析。

**Agent 架构**（对应讲义理论回顾）：

| Agent 组件 | 本 Day 实现 | 营销映射 |
|-----------|------------|---------|
| 感知（Perception） | 用户输入营销任务 | "为一款护肤品制定策略" |
| 工具（Action） | `@tool` 定义三个真实工具 | ROI计算 / 情感分析 / 策略写入 |
| 规划（Planning） | `create_react_agent`（ReAct模式） | Thought-Action-Observation 循环 |
| 记忆（Memory） | `MemorySaver`（短期记忆） | 多轮对话上下文 |
| 反思（Reflection） | Evaluator-Optimizer 循环 | 策略质量自检 |

**三个真实工具**：
- `calculate_roi(revenue, cost)`：计算营销投资回报率
- `analyze_sentiment(text)`：基于关键词分析中文文本情感
- `write_strategy(filename, content)`：将策略写入文件

**营销数据**（内嵌于任务描述，无需外部文件）：
- 产品：透肌焕亮精华液，售价299元，成本80元
- 预计月销2000件，月营销预算50000元
- 用户评价样本："效果好，推荐！"、"价格贵，效果一般"

In [ ]:
import os
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
import warnings
warnings.filterwarnings('ignore')

## 1：定义真实工具

In [ ]:
# 1. 定义三个真实工具
@tool
def calculate_roi(revenue: float, cost: float) -> str:
    """计算营销投资回报率（ROI）。
    参数：revenue - 总收入（元）；cost - 总成本（元）
    返回：ROI 百分比"""
    roi = (revenue - cost) / cost * 100
    return f"ROI = {roi:.1f}%"

@tool
def analyze_sentiment(text: str) -> str:
    """分析中文文本的情感倾向。
    参数：text - 待分析文本
    返回：情感得分和倾向（正面/负面/中性）"""
    positive_words = ["好", "棒", "喜欢", "推荐", "优质", "明显", "光滑", "回购", "满意"]
    negative_words = ["差", "糟", "失望", "投诉", "劣质", "贵", "慢", "一般", "不满意"]
    score = sum(1 for w in positive_words if w in text) - sum(1 for w in negative_words if w in text)
    tendency = "正面" if score > 0 else "负面" if score < 0 else "中性"
    return f"情感得分: {score}（{tendency}）"

@tool
def write_strategy(filename: str, content: str) -> str:
    """将营销策略写入文件。
    参数：filename - 文件路径；content - 策略内容
    返回：写入确认信息"""
    with open(filename, 'w', encoding='utf-8') as f:
        f.write(content)
    return f"策略已写入 {filename}（{len(content)} 字）"

# 测试工具（不需要 LLM，直接调用）
print("=== 工具测试 ===")
print(calculate_roi.invoke({"revenue": 10000, "cost": 4000}))
print(analyze_sentiment.invoke({"text": "这款产品效果好，推荐！但价格有点贵。"}))
print(write_strategy.invoke({"filename": "/tmp/test_strategy.txt", "content": "测试策略"}))

# 营销映射：这三个工具对应营销Agent的"行动"组件
# calculate_roi -> 投入产出分析
# analyze_sentiment -> 用户评价分析
# write_strategy -> 策略输出归档

## 2. ReAct 模式回顾

ReAct（Reasoning + Acting）= Thought-Action-Observation 交替循环：

```
Thought: 用户要推广护肤品，我需要先计算ROI
Action: calculate_roi(revenue=598000, cost=66000)
Observation: ROI = 806.1%
Thought: ROI很高，接下来分析用户评价
Action: analyze_sentiment("效果好，推荐！")
Observation: 情感得分: 2（正面）
Thought: 评价正面，现在写策略
Action: write_strategy("strategy.txt", "...")
Observation: 策略已写入
```

LangGraph 的 `create_react_agent(model, tools)` 自动实现这个循环：
1. LLM 决定调用哪个工具（Thought -> Action）
2. 框架执行工具，返回结果（Observation）
3. LLM 基于结果决定下一步
4. 循环直到 LLM 认为任务完成，输出最终回复

## 2-3：构建并运行 ReAct Agent

In [ ]:
# 2. 构建 ReAct Agent
from langchain_openai import ChatOpenAI

# 方案 A：OpenAI（取消下行注释并填入你的 key）
# os.environ["OPENAI_API_KEY"] = "sk-..."
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 方案 B：Anthropic Claude（推荐用于本课程，需 pip install langchain-anthropic）
# from langchain_anthropic import ChatAnthropic
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# model = ChatAnthropic(model="claude-sonnet-4-20250514")

tools = [calculate_roi, analyze_sentiment, write_strategy]
agent = create_react_agent(model, tools)

print(f"ReAct Agent 构建完成！")
print(f"已绑定 {len(tools)} 个工具：{[t.name for t in tools]}")
print(f"模型：{model.__class__.__name__}")

In [ ]:
# 3. 运行 Agent 处理营销任务
task = """为一款护肤品制定营销策略：
- 产品名：透肌焕亮精华液
- 售价：299元，成本：80元
- 预计月销2000件，月营销预算50000元
- 用户评价："效果好，推荐！"、"价格贵，效果一般"

请：1) 计算预计ROI  2) 分析用户评价情感  3) 将策略写入 /tmp/marketing_strategy.txt
"""

result = agent.invoke({
    "messages": [{"role": "user", "content": task}]
})

# 打印 Agent 推理过程（Thought-Action-Observation 循环）
for msg in result["messages"]:
    print(f"--- {msg.type.upper()} ---")
    print(str(msg.content)[:300])
    print()

## 3. Agent 记忆（Memory）

**短期记忆**：当前对话的 messages 列表。没有记忆的 Agent 每次调用都是独立的--它不记得上一轮说了什么。

LangGraph 用 **checkpointer** 实现短期记忆：
- `MemorySaver()`：内存中的 checkpointer（重启丢失，适合开发）
- `thread_id`：会话标识，不同 thread_id 的对话互不干扰

```python
# 无记忆：每次独立
agent.invoke({"messages": [...]})

# 有记忆：同一 thread_id 共享上下文
agent_mem.invoke(
    {"messages": [...]},
    config={"configurable": {"thread_id": "session-1"}}
)
```

**营销场景**：用户第一轮说"产品售价299"，第二轮说"算一下ROI"--有记忆的 Agent 能理解"算ROI"指的是刚才提到的产品。

## 4：添加短期记忆

In [ ]:
# 4. 用 MemorySaver 添加短期记忆
agent_mem = create_react_agent(model, tools, checkpointer=MemorySaver())

# 第一轮对话：提供产品信息
agent_mem.invoke(
    {"messages": [{"role": "user", "content": "我要为一款护肤品做营销，产品名透肌焕亮精华液，售价299元，成本80元。"}]},
    config={"configurable": {"thread_id": "session-1"}}
)

# 第二轮对话：引用第一轮的上下文（Agent 应记住产品信息）
result_mem = agent_mem.invoke(
    {"messages": [{"role": "user", "content": "基于刚才的产品信息，计算月销2000件的ROI，并分析评价'效果好推荐'和'价格贵效果一般'的情感。"}]},
    config={"configurable": {"thread_id": "session-1"}}
)

# 打印第二轮对话结果
final_msg = result_mem["messages"][-1]
print("第二轮 Agent 回复：")
print(str(final_msg.content)[:500])

## 4. Reflection（反思）模式

Reflection = 生成 -> 评估 -> 改进循环。对应 Anthropic 五模式中的 **Evaluator-Optimizer**。

```
生成者：产出策略初稿
评估者："品牌调性不一致，缺少数据支撑"
生成者：修改为改进版
```

**实现方式**：用三次独立的 LLM 调用：
1. 生成者 LLM：产出初稿
2. 评估者 LLM：审查初稿，指出问题
3. 生成者 LLM：根据反馈改进

**注意**：Reflection 增加 token 消耗和延迟，适合质量敏感任务（如最终交付的营销策略）。

## 5：Reflection 循环

In [ ]:
# 5. 实现 Reflection（评估者-优化者）循环

# 步骤1：生成者 - 产出初稿
draft_task = "为一款护肤品（透肌焕亮精华液，售价299元）写一段100字的营销策略。"
draft_response = model.invoke(draft_task)
draft = draft_response.content

# 步骤2：评估者 - 审查初稿
eval_prompt = f"""你是营销策略评审专家。评估以下策略的质量，指出至少2个问题：

策略：{draft}

请指出问题并给出改进建议。"""
critique_response = model.invoke(eval_prompt)
critique = critique_response.content

# 步骤3：优化者 - 根据反馈改进
improve_prompt = f"""根据以下反馈，改进营销策略：

原策略：{draft}

评审反馈：{critique}

请输出改进后的策略（100字以内）。"""
final_response = model.invoke(improve_prompt)
final = final_response.content

reflection_result = {"draft": draft, "critique": critique, "final": final}

print("=== Reflection 循环 ===")
print(f"初稿：{reflection_result['draft'][:200]}...")
print(f"\n评估：{reflection_result['critique'][:200]}...")
print(f"\n改进版：{reflection_result['final'][:200]}...")

## 5. Plan-Execute 模式

Plan-Execute = 先规划再执行。与 ReAct 的"边想边做"不同，Plan-Execute 先让 LLM 制定完整计划，再逐步执行。

```
Planner: 1.分析目标人群 2.竞品分析 3.生成策略
Executor: 用 ReAct Agent 逐步执行每个步骤
Synthesizer: 汇总各步结果
```

**优势**：全局视角，执行更可控。
**劣势**：计划是静态的，如果中途发现前提错误，需要重新规划。

**实现方式**：
1. Planner LLM：将任务分解为步骤列表
2. Executor（ReAct Agent）：逐步执行
3. Synthesizer LLM：汇总各步结果

## 6（可选）：Plan-Execute 模式

In [ ]:
# 6（可选）：Plan-Execute 模式

# 步骤1：Planner - 制定计划
plan_prompt = """你是营销策划专家。将以下任务分解为3个可执行步骤，每步用数字编号：
任务：为一款护肤品（透肌焕亮精华液，售价299元，成本80元）制定营销策略。
只输出步骤列表，不要额外解释。"""
plan_response = model.invoke(plan_prompt)
plan = plan_response.content

# 步骤2：Executor - 用 ReAct Agent 逐步执行
steps = [line.strip() for line in plan.split('\n') if line.strip() and line[0].isdigit()]
step_results = []

for i, step in enumerate(steps):
    print(f"执行步骤 {i+1}/{len(steps)}: {step[:50]}...")
    step_result = agent.invoke({
        "messages": [{"role": "user", "content": f"执行：{step}（产品：透肌焕亮精华液，售价299，成本80）"}]
    })
    step_results.append(step_result["messages"][-1].content)

# 步骤3：Synthesizer - 汇总最终结果
final_prompt = f"""基于以下各步骤结果，综合成一份完整的营销策略：

{chr(10).join(f'步骤{i+1}：{r}' for i, r in enumerate(step_results))}"""
final_response = model.invoke(final_prompt)

plan_result = {"plan": plan, "steps": step_results, "final": final_response.content}

print("=== Plan-Execute 模式 ===")
print(f"计划：\n{plan_result['plan']}")
print(f"\n执行了 {len(plan_result['steps'])} 步")
print(f"\n最终输出：{plan_result['final'][:300]}...")

## 6. 反思与前沿

### 反思问题
1. 你的 ReAct Agent 在处理营销任务时，选择工具的顺序是否符合预期？为什么 LLM 有时会选错工具？（提示：看工具的 docstring 是否清晰）
2. 有记忆 vs 无记忆的 Agent，在多轮对话中行为有什么差异？`thread_id` 的作用是什么？
3. Reflection 循环改进了什么？改进后的策略与初稿的主要差异是什么？
4. Plan-Execute 与 ReAct 在处理同一任务时，路径和结果有什么不同？哪种更适合营销策略生成？

### 2026 前沿：MCP（Model Context Protocol）
MCP 是 Anthropic 2024 年底发布的开放协议，2025-2026 年成为 Agent 互操作的事实标准。它标准化了"AI 应用如何从外部获取上下文和调用工具"：

- **MCP Host**（AI 应用）-> **MCP Client** -> **MCP Server**（工具提供方）
- 三大原语：Tools（可执行函数）、Resources（数据源）、Prompts（交互模板）
- 本 Day 的 `@tool` 是"应用内工具"；MCP 是"进程外工具"（独立 Server 通过 JSON-RPC 协议通信）

生产级 Agent 通常混合使用：简单工具用 `@tool`，企业系统集成（CRM/广告平台/数据分析）用 MCP Server。

参考 https://modelcontextprotocol.io/docs/concepts/architecture